# NotebookLM-Style UI Redesign: Document Groups & Workspaces

## 🎯 Transformation Overview

**From:** Simple "Upload File" screen  
**To:** Workspace-first, group-based document management system

### Key Changes
- ✅ Group/Project-based organization
- ✅ Dashboard with group cards
- ✅ Upload only happens inside groups
- ✅ Clean NotebookLM-style UI
- ✅ localStorage persistence
- ✅ Auto-load last opened group

---

## Architecture at a Glance

```
App (Routes + Context)
├── Dashboard (/)
│   ├── GroupDashboard (list of groups)
│   ├── GroupCard (individual group preview)
│   └── CreateGroupModal (create new group)
└── Workspace (/group/:id)
    ├── GroupPage (main workspace)
    ├── DocumentSidebar (docs list + upload)
    └── ChatPanel (RAG interface)
```

## 1. Data Models & TypeScript Types

Define the core data structures that power the new workspace system.

In [ ]:
# File: src/types/workspace.ts

typescript_types = '''
// Document type - individual files within a group
export interface Document {
  id: string;
  groupId: string;
  name: string;
  fileSize: number;
  fileHash: string;
  uploadedAt: Date;
  fileType: 'pdf' | 'txt' | 'md' | 'csv';
  metadata?: {
    pageCount?: number;
    chunkCount?: number;
  };
}

// Group/Project type - container for related documents
export interface Group {
  id: string;
  name: string;
  description?: string;
  createdAt: Date;
  updatedAt?: Date;
  documents: Document[];
  metadata?: {
    totalChunks?: number;
    totalSize?: number;
    lastAccessed?: Date;
  };
}

// State for GroupContext
export interface GroupContextState {
  groups: Group[];
  currentGroupId: string | null;
  loading: boolean;
  error: string | null;
}

// Actions for context reducer
export type GroupAction =
  | { type: 'SET_GROUPS'; payload: Group[] }
  | { type: 'CREATE_GROUP'; payload: Group }
  | { type: 'UPDATE_GROUP'; payload: Group }
  | { type: 'DELETE_GROUP'; payload: string }
  | { type: 'SET_CURRENT_GROUP'; payload: string | null }
  | { type: 'ADD_DOCUMENT'; payload: { groupId: string; document: Document } }
  | { type: 'REMOVE_DOCUMENT'; payload: { groupId: string; documentId: string } }
  | { type: 'SET_LOADING'; payload: boolean }
  | { type: 'SET_ERROR'; payload: string | null };
'''

print("TypeScript Interfaces for Workspace System:")
print(typescript_types)

## 2. Group Context & State Management

Central state management for groups and documents using React Context API.

In [ ]:
# File: src/context/GroupContext.tsx
context_code = '''
import React, { createContext, useReducer, useEffect, ReactNode } from 'react';
import { Group, Document } from '../types/workspace';

// Context type definitions
interface GroupContextState {
  groups: Group[];
  currentGroupId: string | null;
  loading: boolean;
  error: string | null;
}

type GroupAction =
  | { type: 'SET_GROUPS'; payload: Group[] }
  | { type: 'CREATE_GROUP'; payload: Group }
  | { type: 'UPDATE_GROUP'; payload: Group }
  | { type: 'DELETE_GROUP'; payload: string }
  | { type: 'SET_CURRENT_GROUP'; payload: string }
  | { type: 'ADD_DOCUMENT'; payload: { groupId: string; document: Document } }
  | { type: 'UPDATE_DOCUMENT'; payload: { groupId: string; document: Document } }
  | { type: 'DELETE_DOCUMENT'; payload: { groupId: string; documentId: string } }
  | { type: 'SET_LOADING'; payload: boolean }
  | { type: 'SET_ERROR'; payload: string | null };

const initialState: GroupContextState = {
  groups: [],
  currentGroupId: null,
  loading: false,
  error: null,
};

// Reducer function for managing group state
const groupReducer = (state: GroupContextState, action: GroupAction): GroupContextState => {
  switch (action.type) {
    case 'SET_GROUPS':
      return { ...state, groups: action.payload };
    
    case 'CREATE_GROUP':
      return { 
        ...state, 
        groups: [...state.groups, action.payload],
        currentGroupId: action.payload.id 
      };
    
    case 'UPDATE_GROUP':
      return {
        ...state,
        groups: state.groups.map(g => g.id === action.payload.id ? action.payload : g)
      };
    
    case 'DELETE_GROUP':
      const remaining = state.groups.filter(g => g.id !== action.payload);
      return {
        ...state,
        groups: remaining,
        currentGroupId: state.currentGroupId === action.payload 
          ? (remaining.length > 0 ? remaining[0].id : null)
          : state.currentGroupId
      };
    
    case 'SET_CURRENT_GROUP':
      return { ...state, currentGroupId: action.payload };
    
    case 'ADD_DOCUMENT':
      return {
        ...state,
        groups: state.groups.map(g =>
          g.id === action.payload.groupId
            ? { ...g, documents: [...g.documents, action.payload.document] }
            : g
        )
      };
    
    case 'UPDATE_DOCUMENT':
      return {
        ...state,
        groups: state.groups.map(g =>
          g.id === action.payload.groupId
            ? {
                ...g,
                documents: g.documents.map(d =>
                  d.id === action.payload.document.id ? action.payload.document : d
                )
              }
            : g
        )
      };
    
    case 'DELETE_DOCUMENT':
      return {
        ...state,
        groups: state.groups.map(g =>
          g.id === action.payload.groupId
            ? { ...g, documents: g.documents.filter(d => d.id !== action.payload.documentId) }
            : g
        )
      };
    
    case 'SET_LOADING':
      return { ...state, loading: action.payload };
    
    case 'SET_ERROR':
      return { ...state, error: action.payload };
    
    default:
      return state;
  }
};

// Context creation
export const GroupContext = createContext<{
  state: GroupContextState;
  dispatch: React.Dispatch<GroupAction>;
}>({
  state: initialState,
  dispatch: () => {}
});

// Provider component
interface GroupProviderProps {
  children: ReactNode;
}

export const GroupProvider: React.FC<GroupProviderProps> = ({ children }) => {
  const [state, dispatch] = useReducer(groupReducer, initialState);

  // Load groups from localStorage on mount
  useEffect(() => {
    const loadGroups = () => {
      try {
        dispatch({ type: 'SET_LOADING', payload: true });
        const savedGroups = localStorage.getItem('workspace_groups');
        
        if (savedGroups) {
          const parsed: Group[] = JSON.parse(savedGroups);
          dispatch({ type: 'SET_GROUPS', payload: parsed });
          
          // Auto-load last opened group
          const lastGroupId = localStorage.getItem('last_group_id');
          if (lastGroupId && parsed.some(g => g.id === lastGroupId)) {
            dispatch({ type: 'SET_CURRENT_GROUP', payload: lastGroupId });
          } else if (parsed.length > 0) {
            dispatch({ type: 'SET_CURRENT_GROUP', payload: parsed[0].id });
          }
        }
        dispatch({ type: 'SET_LOADING', payload: false });
      } catch (error) {
        dispatch({ type: 'SET_ERROR', payload: 'Failed to load groups' });
        dispatch({ type: 'SET_LOADING', payload: false });
      }
    };

    loadGroups();
  }, []);

  // Persist groups to localStorage whenever they change
  useEffect(() => {
    localStorage.setItem('workspace_groups', JSON.stringify(state.groups));
  }, [state.groups]);

  // Persist current group ID
  useEffect(() => {
    if (state.currentGroupId) {
      localStorage.setItem('last_group_id', state.currentGroupId);
    }
  }, [state.currentGroupId]);

  return (
    <GroupContext.Provider value={{ state, dispatch }}>
      {children}
    </GroupContext.Provider>
  );
};

// Custom hook for using the context
export const useGroupContext = () => {
  const context = React.useContext(GroupContext);
  if (!context) {
    throw new Error('useGroupContext must be used within GroupProvider');
  }
  return context;
};
'''

print(context_code)

## 3. Group Dashboard Component

The home screen displaying all workspace groups with ability to create, search, and access groups.

In [ ]:
# File: src/components/GroupDashboard.tsx
dashboard_code = '''
import React, { useState } from 'react';
import { useNavigate } from 'react-router-dom';
import { Plus, Search, Trash2 } from 'lucide-react';
import { motion } from 'framer-motion';
import { useGroupContext } from '../context/GroupContext';
import { CreateGroupModal } from './CreateGroupModal';
import GroupCard from './GroupCard';

export const GroupDashboard: React.FC = () => {
  const navigate = useNavigate();
  const { state, dispatch } = useGroupContext();
  const [searchQuery, setSearchQuery] = useState('');
  const [showCreateModal, setShowCreateModal] = useState(false);

  // Filter groups based on search
  const filteredGroups = state.groups.filter(group =>
    group.name.toLowerCase().includes(searchQuery.toLowerCase()) ||
    group.description?.toLowerCase().includes(searchQuery.toLowerCase())
  );

  // Handle group selection
  const handleGroupSelect = (groupId: string) => {
    dispatch({ type: 'SET_CURRENT_GROUP', payload: groupId });
    navigate(`/group/${groupId}`);
  };

  // Handle group deletion
  const handleGroupDelete = (groupId: string, e: React.MouseEvent) => {
    e.stopPropagation();
    if (window.confirm('Are you sure you want to delete this group?')) {
      dispatch({ type: 'DELETE_GROUP', payload: groupId });
    }
  };

  return (
    <div className="min-h-screen bg-gradient-to-br from-slate-50 to-slate-100">
      {/* Header */}
      <div className="border-b border-slate-200 bg-white">
        <div className="max-w-7xl mx-auto px-6 py-8">
          <motion.div
            initial={{ opacity: 0, y: -10 }}
            animate={{ opacity: 1, y: 0 }}
            className="flex justify-between items-center"
          >
            <div>
              <h1 className="text-4xl font-bold text-slate-900">Workspace</h1>
              <p className="text-slate-600 mt-1">Organize and analyze your documents in groups</p>
            </div>
            <button
              onClick={() => setShowCreateModal(true)}
              className="flex items-center gap-2 px-4 py-3 bg-blue-600 text-white rounded-lg hover:bg-blue-700 transition-colors shadow-lg hover:shadow-xl"
            >
              <Plus size={20} />
              New Group
            </button>
          </motion.div>
        </div>
      </div>

      {/* Search Bar */}
      <div className="max-w-7xl mx-auto px-6 py-6">
        <div className="relative">
          <Search className="absolute left-4 top-1/2 transform -translate-y-1/2 text-slate-400" size={20} />
          <input
            type="text"
            placeholder="Search groups..."
            value={searchQuery}
            onChange={(e) => setSearchQuery(e.target.value)}
            className="w-full pl-12 pr-4 py-3 rounded-lg border border-slate-200 focus:border-blue-500 focus:outline-none focus:ring-2 focus:ring-blue-500/20 transition-all"
          />
        </div>
      </div>

      {/* Groups Grid */}
      <div className="max-w-7xl mx-auto px-6 pb-12">
        {filteredGroups.length === 0 ? (
          <motion.div
            initial={{ opacity: 0 }}
            animate={{ opacity: 1 }}
            className="text-center py-16"
          >
            <div className="text-slate-400 text-6xl mb-4">📁</div>
            <h3 className="text-xl font-semibold text-slate-700 mb-2">
              {state.groups.length === 0 ? 'No groups yet' : 'No groups match your search'}
            </h3>
            <p className="text-slate-600">
              {state.groups.length === 0
                ? 'Create your first group to get started'
                : 'Try a different search term'}
            </p>
            {state.groups.length === 0 && (
              <button
                onClick={() => setShowCreateModal(true)}
                className="mt-6 px-6 py-2 bg-blue-600 text-white rounded-lg hover:bg-blue-700 transition-colors"
              >
                Create First Group
              </button>
            )}
          </motion.div>
        ) : (
          <motion.div
            initial={{ opacity: 0 }}
            animate={{ opacity: 1 }}
            className="grid grid-cols-1 md:grid-cols-2 lg:grid-cols-3 gap-6"
          >
            {filteredGroups.map((group, index) => (
              <motion.div
                key={group.id}
                initial={{ opacity: 0, y: 20 }}
                animate={{ opacity: 1, y: 0 }}
                transition={{ delay: index * 0.1 }}
              >
                <GroupCard
                  group={group}
                  onClick={() => handleGroupSelect(group.id)}
                  onDelete={(e) => handleGroupDelete(group.id, e)}
                />
              </motion.div>
            ))}
          </motion.div>
        )}
      </div>

      {/* Create Group Modal */}
      <CreateGroupModal
        isOpen={showCreateModal}
        onClose={() => setShowCreateModal(false)}
      />
    </div>
  );
};
'''

print(dashboard_code)

## 4. Group Card Component

Individual card displaying group information with quick actions.

In [ ]:
# File: src/components/GroupCard.tsx
card_code = '''
import React from 'react';
import { motion } from 'framer-motion';
import { FileText, Trash2, ChevronRight } from 'lucide-react';
import { Group } from '../types/workspace';
import { formatDistanceToNow } from 'date-fns';

interface GroupCardProps {
  group: Group;
  onClick: () => void;
  onDelete: (e: React.MouseEvent) => void;
}

const GroupCard: React.FC<GroupCardProps> = ({ group, onClick, onDelete }) => {
  const documentCount = group.documents.length;
  const createdDate = new Date(group.createdAt);

  return (
    <motion.div
      whileHover={{ y: -4, boxShadow: '0 20px 25px -5px rgba(0, 0, 0, 0.1)' }}
      whileTap={{ scale: 0.98 }}
      onClick={onClick}
      className="cursor-pointer batch-card bg-white rounded-lg border border-slate-200 p-6 transition-all hover:border-blue-300"
    >
      {/* Header with icon and menu */}
      <div className="flex justify-between items-start mb-4">
        <div className="flex-1">
          <h3 className="text-lg font-semibold text-slate-900 mb-1">{group.name}</h3>
          {group.description && (
            <p className="text-sm text-slate-600 line-clamp-2">{group.description}</p>
          )}
        </div>

        {/* Delete button */}
        <motion.button
          whileHover={{ scale: 1.1 }}
          whileTap={{ scale: 0.95 }}
          onClick={onDelete}
          className="p-2 text-slate-400 hover:text-red-600 hover:bg-red-50 rounded transition-colors ml-2"
        >
          <Trash2 size={18} />
        </motion.button>
      </div>

      {/* Divider */}
      <div className="h-px bg-slate-100 mb-4" />

      {/* Footer with document count and date */}
      <div className="flex justify-between items-center">
        <div className="flex items-center gap-2 text-sm text-slate-600">
          <FileText size={16} className="text-blue-500" />
          <span>{documentCount} document{documentCount !== 1 ? 's' : ''}</span>
        </div>

        <div className="flex items-center gap-2 text-slate-500 hover:text-blue-600 transition-colors">
          <span className="text-xs">
            {formatDistanceToNow(createdDate, { addSuffix: true })}
          </span>
          <ChevronRight size={16} />
        </div>
      </div>
    </motion.div>
  );
};

export default GroupCard;
'''

print(card_code)

## 5. Create Group Modal

Modal dialog for creating new groups with name and optional description.

In [ ]:
# File: src/components/CreateGroupModal.tsx
modal_code = '''
import React, { useState } from 'react';
import { motion } from 'framer-motion';
import { X } from 'lucide-react';
import { useGroupContext } from '../context/GroupContext';
import { Group } from '../types/workspace';

interface CreateGroupModalProps {
  isOpen: boolean;
  onClose: () => void;
}

export const CreateGroupModal: React.FC<CreateGroupModalProps> = ({ isOpen, onClose }) => {
  const { dispatch } = useGroupContext();
  const [formData, setFormData] = useState({
    name: '',
    description: ''
  });
  const [error, setError] = useState('');
  const [isSubmitting, setIsSubmitting] = useState(false);

  const handleSubmit = async (e: React.FormEvent) => {
    e.preventDefault();
    setError('');

    // Validation
    if (!formData.name.trim()) {
      setError('Group name is required');
      return;
    }

    if (formData.name.trim().length < 3) {
      setError('Group name must be at least 3 characters');
      return;
    }

    try {
      setIsSubmitting(true);

      // Create new group
      const newGroup: Group = {
        id: `group_${Date.now()}_${Math.random().toString(36).substr(2, 9)}`,
        name: formData.name.trim(),
        description: formData.description.trim() || undefined,
        createdAt: new Date().toISOString(),
        documents: [],
        metadata: {}
      };

      dispatch({ type: 'CREATE_GROUP', payload: newGroup });

      // Reset and close
      setFormData({ name: '', description: '' });
      onClose();
    } catch (err) {
      setError('Failed to create group. Please try again.');
    } finally {
      setIsSubmitting(false);
    }
  };

  if (!isOpen) return null;

  return (
    <motion.div
      initial={{ opacity: 0 }}
      animate={{ opacity: 1 }}
      exit={{ opacity: 0 }}
      onClick={onClose}
      className="fixed inset-0 bg-black/50 flex items-center justify-center z-50 p-4"
    >
      <motion.div
        initial={{ scale: 0.95, opacity: 0 }}
        animate={{ scale: 1, opacity: 1 }}
        exit={{ scale: 0.95, opacity: 0 }}
        onClick={(e) => e.stopPropagation()}
        className="bg-white rounded-lg max-w-md w-full shadow-2xl"
      >
        {/* Header */}
        <div className="flex justify-between items-center p-6 border-b border-slate-200">
          <h2 className="text-xl font-bold text-slate-900">Create New Group</h2>
          <button
            onClick={onClose}
            className="p-1 hover:bg-slate-100 rounded transition-colors"
          >
            <X size={20} className="text-slate-500" />
          </button>
        </div>

        {/* Form */}
        <form onSubmit={handleSubmit} className="p-6">
          {/* Name Input */}
          <div className="mb-4">
            <label className="block text-sm font-medium text-slate-700 mb-2">
              Group Name *
            </label>
            <input
              type="text"
              value={formData.name}
              onChange={(e) => setFormData({ ...formData, name: e.target.value })}
              placeholder="e.g., Project Alpha"
              className="w-full px-4 py-2 border border-slate-200 rounded-lg focus:border-blue-500 focus:outline-none focus:ring-2 focus:ring-blue-500/20 transition-all"
              disabled={isSubmitting}
            />
          </div>

          {/* Description Input */}
          <div className="mb-4">
            <label className="block text-sm font-medium text-slate-700 mb-2">
              Description (Optional)
            </label>
            <textarea
              value={formData.description}
              onChange={(e) => setFormData({ ...formData, description: e.target.value })}
              placeholder="Brief description of this group..."
              rows={3}
              className="w-full px-4 py-2 border border-slate-200 rounded-lg focus:border-blue-500 focus:outline-none focus:ring-2 focus:ring-blue-500/20 transition-all resize-none"
              disabled={isSubmitting}
            />
          </div>

          {/* Error Message */}
          {error && (
            <motion.div
              initial={{ opacity: 0, y: -10 }}
              animate={{ opacity: 1, y: 0 }}
              className="mb-4 p-3 bg-red-50 border border-red-200 rounded-lg text-sm text-red-700"
            >
              {error}
            </motion.div>
          )}

          {/* Submit Button */}
          <button
            type="submit"
            disabled={isSubmitting}
            className="w-full px-4 py-2 bg-blue-600 text-white font-medium rounded-lg hover:bg-blue-700 disabled:bg-slate-400 transition-colors"
          >
            {isSubmitting ? 'Creating...' : 'Create Group'}
          </button>
        </form>
      </motion.div>
    </motion.div>
  );
};
'''

print(modal_code)

## 6. Group Page Component

Main workspace interface with two-panel layout: document sidebar + chat panel.

In [ ]:
# File: src/components/GroupPage.tsx
group_page_code = '''
import React, { useState, useEffect } from 'react';
import { useParams, useNavigate } from 'react-router-dom';
import { ChevronLeft, Upload, Trash2, Menu } from 'lucide-react';
import { motion } from 'framer-motion';
import { useGroupContext } from '../context/GroupContext';
import { ChatEngine } from '../services/ChatEngine';
import { DocumentProcessor } from '../services/DocumentProcessor';
import DocumentSidebar from './DocumentSidebar';
import ChatPanel from './ChatPanel';

export const GroupPage: React.FC = () => {
  const { groupId } = useParams<{ groupId: string }>();
  const navigate = useNavigate();
  const { state, dispatch } = useGroupContext();
  const [sidebarOpen, setSidebarOpen] = useState(true);
  const [isProcessing, setIsProcessing] = useState(false);

  // Get current group
  const currentGroup = state.groups.find(g => g.id === groupId);

  // Set current group in context
  useEffect(() => {
    if (groupId) {
      dispatch({ type: 'SET_CURRENT_GROUP', payload: groupId });
    }
  }, [groupId, dispatch]);

  // Handle file upload
  const handleFileUpload = async (files: FileList) => {
    if (!currentGroup || !files) return;

    try {
      setIsProcessing(true);

      for (let i = 0; i < files.length; i++) {
        const file = files[i];

        // Process file
        const { text: content, images } = await DocumentProcessor.extractContent(file);

        // Create document record
        const fileHash = await DocumentProcessor.hashFile(file);
        const newDocument = {
          id: `doc_${Date.now()}_${i}`,
          groupId: currentGroup.id,
          name: file.name,
          fileSize: file.size,
          fileHash,
          uploadedAt: new Date().toISOString(),
          fileType: file.type,
          metadata: {
            pages: images.length,
            uploadedBy: 'user'
          }
        };

        // Add to group
        dispatch({
          type: 'ADD_DOCUMENT',
          payload: {
            groupId: currentGroup.id,
            document: newDocument
          }
        });

        // Process with ChatEngine (embeddings)
        await ChatEngine.processFile(file);
      }
    } catch (error) {
      console.error('File upload error:', error);
      dispatch({
        type: 'SET_ERROR',
        payload: 'Failed to upload file. Please try again.'
      });
    } finally {
      setIsProcessing(false);
    }
  };

  if (!currentGroup) {
    return (
      <div className="h-screen flex items-center justify-center">
        <div className="text-center">
          <p className="text-slate-600 mb-4">Group not found</p>
          <button
            onClick={() => navigate('/')}
            className="px-6 py-2 bg-blue-600 text-white rounded-lg hover:bg-blue-700"
          >
            Back to Dashboard
          </button>
        </div>
      </div>
    );
  }

  return (
    <div className="h-screen flex flex-col bg-slate-50">
      {/* Header */}
      <div className="bg-white border-b border-slate-200 px-6 py-4 flex items-center justify-between">
        <div className="flex items-center gap-4">
          <button
            onClick={() => navigate('/')}
            className="p-2 hover:bg-slate-100 rounded transition-colors"
          >
            <ChevronLeft size={20} className="text-slate-600" />
          </button>
          <div>
            <h1 className="text-2xl font-bold text-slate-900">{currentGroup.name}</h1>
            {currentGroup.description && (
              <p className="text-sm text-slate-600">{currentGroup.description}</p>
            )}
          </div>
        </div>

        {/* Mobile menu toggle */}
        <button
          onClick={() => setSidebarOpen(!sidebarOpen)}
          className="lg:hidden p-2 hover:bg-slate-100 rounded transition-colors"
        >
          <Menu size={20} />
        </button>
      </div>

      {/* Main Content */}
      <div className="flex flex-1 overflow-hidden">
        {/* Document Sidebar */}
        <motion.div
          animate={{
            width: sidebarOpen ? 'auto' : 0,
            opacity: sidebarOpen ? 1 : 0,
            pointerEvents: sidebarOpen ? 'auto' : 'none'
          }}
          className="hidden lg:flex w-72 border-r border-slate-200 bg-white overflow-hidden"
        >
          <DocumentSidebar
            documents={currentGroup.documents || []}
            onFileUpload={handleFileUpload}
            isProcessing={isProcessing}
          />
        </motion.div>

        {/* Mobile Sidebar Overlay */}
        {sidebarOpen && (
          <motion.div
            initial={{ opacity: 0 }}
            animate={{ opacity: 1 }}
            exit={{ opacity: 0 }}
            onClick={() => setSidebarOpen(false)}
            className="lg:hidden fixed inset-0 bg-black/50 z-40"
          />
        )}

        {/* Chat Panel */}
        <div className="flex-1 flex flex-col overflow-hidden">
          <ChatPanel
            groupId={currentGroup.id}
            documents={currentGroup.documents || []}
          />
        </div>
      </div>
    </div>
  );
};
'''

print(group_page_code)

## 7. Document Sidebar Component

Left panel showing uploaded documents with upload area and file management.

In [ ]:
# File: src/components/DocumentSidebar.tsx
sidebar_code = '''
import React, { useState } from 'react';
import { Upload, FileText, Trash2, Eye } from 'lucide-react';
import { motion } from 'framer-motion';
import { useDropzone } from 'react-dropzone';
import { Document } from '../types/workspace';
import { formatBytes } from '../utils/format';

interface DocumentSidebarProps {
  documents: Document[];
  onFileUpload: (files: FileList) => void;
  isProcessing: boolean;
}

const DocumentSidebar: React.FC<DocumentSidebarProps> = ({
  documents,
  onFileUpload,
  isProcessing
}) => {
  const [selectedDocId, setSelectedDocId] = useState<string | null>(null);

  const { getRootProps, getInputProps, isDragActive } = useDropzone({
    onDrop: (files) => {
      const fileList = new DataTransfer();
      files.forEach(file => fileList.items.add(file));
      onFileUpload(fileList.files);
    },
    accept: {
      'application/pdf': ['.pdf'],
      'text/plain': ['.txt'],
      'text/markdown': ['.md'],
      'text/csv': ['.csv']
    }
  });

  const supportedFormats = ['PDF', 'TXT', 'Markdown', 'CSV'];

  return (
    <div className="w-full h-full flex flex-col">
      {/* Upload Area */}
      <div className="p-4 border-b border-slate-200">
        <div
          {...getRootProps()}
          className={`p-4 border-2 border-dashed rounded-lg transition-all cursor-pointer ${
            isDragActive
              ? 'border-blue-500 bg-blue-50'
              : 'border-slate-300 hover:border-slate-400'
          }`}
        >
          <input {...getInputProps()} disabled={isProcessing} />
          <div className="flex flex-col items-center gap-2 text-sm">
            <Upload size={20} className={isDragActive ? 'text-blue-500' : 'text-slate-400'} />
            <div className="text-center">
              <p className="font-medium text-slate-900">
                {isDragActive ? 'Drop files here' : 'Drop files to upload'}
              </p>
              <p className="text-xs text-slate-500">or click to select</p>
            </div>
          </div>
        </div>

        {/* Supported formats info */}
        <p className="text-xs text-slate-500 mt-3 text-center">
          Supports: {supportedFormats.join(', ')}
        </p>
      </div>

      {/* Documents List */}
      <div className="flex-1 overflow-y-auto p-4">
        {documents.length === 0 ? (
          <div className="text-center py-8">
            <FileText size={32} className="text-slate-300 mx-auto mb-3" />
            <p className="text-sm text-slate-500">No documents yet</p>
            <p className="text-xs text-slate-400">Upload documents to get started</p>
          </div>
        ) : (
          <div className="space-y-2">
            {documents.map((doc, index) => (
              <motion.div
                key={doc.id}
                initial={{ opacity: 0, x: -20 }}
                animate={{ opacity: 1, x: 0 }}
                transition={{ delay: index * 0.05 }}
                className={`p-3 rounded-lg border transition-all cursor-pointer ${
                  selectedDocId === doc.id
                    ? 'bg-blue-50 border-blue-300'
                    : 'bg-slate-50 border-slate-200 hover:bg-slate-100'
                }`}
                onClick={() => setSelectedDocId(doc.id)}
              >
                <div className="flex items-start gap-2">
                  <FileText size={16} className="text-blue-500 mt-0.5 flex-shrink-0" />
                  <div className="flex-1 min-w-0">
                    <h4 className="text-sm font-medium text-slate-900 truncate">
                      {doc.name}
                    </h4>
                    <p className="text-xs text-slate-500">
                      {formatBytes(doc.fileSize)}
                    </p>
                  </div>
                </div>
              </motion.div>
            ))}
          </div>
        )}
      </div>

      {/* Processing indicator */}
      {isProcessing && (
        <div className="p-4 border-t border-slate-200 bg-blue-50">
          <div className="flex items-center gap-2 text-sm text-blue-700">
            <div className="animate-spin">⚙️</div>
            <span>Processing documents...</span>
          </div>
        </div>
      )}
    </div>
  );
};

export default DocumentSidebar;
'''

print(sidebar_code)

## 8. Chat Panel Component

Right panel containing Q&A interface, messages, and action buttons (reuses existing chat logic).

In [ ]:
# File: src/components/ChatPanel.tsx
chat_panel_code = '''
import React, { useState, useRef, useEffect } from 'react';
import { Send, Zap, FileText, Lightbulb, BarChart3 } from 'lucide-react';
import { motion } from 'framer-motion';
import { ChatEngine } from '../services/ChatEngine';
import { Document } from '../types/workspace';
import MessageItem from './MessageItem';

interface ChatPanelProps {
  groupId: string;
  documents: Document[];
}

interface Message {
  id: string;
  role: 'user' | 'assistant';
  content: string;
  timestamp: Date;
  metadata?: {
    type: 'summary' | 'comparison' | 'explanation' | 'keypoints' | 'chat';
  };
}

const ChatPanel: React.FC<ChatPanelProps> = ({ groupId, documents }) => {
  const [messages, setMessages] = useState<Message[]>([]);
  const [input, setInput] = useState('');
  const [isLoading, setIsLoading] = useState(false);
  const [selectedDocuments, setSelectedDocuments] = useState<string[]>([]);
  const messagesEndRef = useRef<HTMLDivElement>(null);

  const scrollToBottom = () => {
    messagesEndRef.current?.scrollIntoView({ behavior: 'smooth' });
  };

  useEffect(() => {
    scrollToBottom();
  }, [messages]);

  // Submit message for Q&A
  const handleSendMessage = async () => {
    if (!input.trim() || isLoading) return;

    const userMessage: Message = {
      id: `msg_${Date.now()}`,
      role: 'user',
      content: input,
      timestamp: new Date(),
      metadata: { type: 'chat' }
    };

    setMessages(prev => [...prev, userMessage]);
    setInput('');
    setIsLoading(true);

    try {
      const response = await ChatEngine.query(input);
      const assistantMessage: Message = {
        id: `msg_${Date.now()}`,
        role: 'assistant',
        content: response,
        timestamp: new Date(),
        metadata: { type: 'chat' }
      };
      setMessages(prev => [...prev, assistantMessage]);
    } catch (error) {
      const errorMessage: Message = {
        id: `msg_${Date.now()}`,
        role: 'assistant',
        content: 'Sorry, I encountered an error processing your request. Please try again.',
        timestamp: new Date()
      };
      setMessages(prev => [...prev, errorMessage]);
    } finally {
      setIsLoading(false);
    }
  };

  // Summarize document
  const handleSummarize = async (length: 'short' | 'medium' | 'long' = 'medium') => {
    setIsLoading(true);
    try {
      const summary = await ChatEngine.summarizeDocument(length);
      const message: Message = {
        id: `msg_${Date.now()}`,
        role: 'assistant',
        content: summary,
        timestamp: new Date(),
        metadata: { type: 'summary' }
      };
      setMessages(prev => [...prev, message]);
    } finally {
      setIsLoading(false);
    }
  };

  // Extract key points
  const handleKeyPoints = async () => {
    setIsLoading(true);
    try {
      const keyPoints = await ChatEngine.extractKeyPoints();
      const message: Message = {
        id: `msg_${Date.now()}`,
        role: 'assistant',
        content: keyPoints,
        timestamp: new Date(),
        metadata: { type: 'keypoints' }
      };
      setMessages(prev => [...prev, message]);
    } finally {
      setIsLoading(false);
    }
  };

  // Compare documents
  const handleCompare = async () => {
    if (documents.length < 2) {
      alert('Please upload at least 2 documents to compare');
      return;
    }

    setIsLoading(true);
    try {
      const fileHashes = documents.map(d => d.fileHash);
      const comparison = await ChatEngine.compareDocuments(fileHashes);
      const message: Message = {
        id: `msg_${Date.now()}`,
        role: 'assistant',
        content: comparison,
        timestamp: new Date(),
        metadata: { type: 'comparison' }
      };
      setMessages(prev => [...prev, message]);
    } finally {
      setIsLoading(false);
    }
  };

  return (
    <div className="flex flex-col h-full bg-white">
      {/* Messages Area */}
      <div className="flex-1 overflow-y-auto p-6 space-y-4">
        {messages.length === 0 ? (
          <motion.div
            initial={{ opacity: 0, y: 20 }}
            animate={{ opacity: 1, y: 0 }}
            className="h-full flex flex-col items-center justify-center text-center"
          >
            <Zap size={48} className="text-blue-500 mb-4" />
            <h2 className="text-2xl font-bold text-slate-900 mb-2">
              Welcome to {documents[0]?.name || 'Your Workspace'}
            </h2>
            <p className="text-slate-600 max-w-md">
              Ask questions about your documents, get summaries, compare documents, or extract key insights.
            </p>

            {/* Quick action cards */}
            <div className="mt-8 grid grid-cols-2 gap-3 max-w-md">
              <motion.button
                whileHover={{ scale: 1.05 }}
                onClick={() => handleSummarize('medium')}
                disabled={isLoading || documents.length === 0}
                className="p-3 bg-blue-50 border border-blue-200 rounded-lg hover:bg-blue-100 transition-colors disabled:opacity-50"
              >
                <Lightbulb size={20} className="text-blue-600 mx-auto mb-1" />
                <p className="text-xs font-medium text-blue-900">Summarize</p>
              </motion.button>

              <motion.button
                whileHover={{ scale: 1.05 }}
                onClick={handleKeyPoints}
                disabled={isLoading || documents.length === 0}
                className="p-3 bg-purple-50 border border-purple-200 rounded-lg hover:bg-purple-100 transition-colors disabled:opacity-50"
              >
                <FileText size={20} className="text-purple-600 mx-auto mb-1" />
                <p className="text-xs font-medium text-purple-900">Key Points</p>
              </motion.button>

              <motion.button
                whileHover={{ scale: 1.05 }}
                onClick={handleCompare}
                disabled={isLoading || documents.length < 2}
                className="p-3 bg-green-50 border border-green-200 rounded-lg hover:bg-green-100 transition-colors disabled:opacity-50"
              >
                <BarChart3 size={20} className="text-green-600 mx-auto mb-1" />
                <p className="text-xs font-medium text-green-900">Compare</p>
              </motion.button>

              <motion.button
                whileHover={{ scale: 1.05 }}
                onClick={() => setInput('What are the main topics covered?')}
                className="p-3 bg-orange-50 border border-orange-200 rounded-lg hover:bg-orange-100 transition-colors"
              >
                <Zap size={20} className="text-orange-600 mx-auto mb-1" />
                <p className="text-xs font-medium text-orange-900">Ask</p>
              </motion.button>
            </div>
          </motion.div>
        ) : (
          <div>
            {messages.map((message, index) => (
              <MessageItem
                key={message.id}
                message={message}
                isLast={index === messages.length - 1}
              />
            ))}
            {isLoading && (
              <motion.div
                initial={{ opacity: 0 }}
                animate={{ opacity: 1 }}
                className="flex items-center gap-2 p-4 bg-slate-100 rounded-lg max-w-xs"
              >
                <div className="animate-spin">⚙️</div>
                <span className="text-sm text-slate-700">AI is thinking...</span>
              </motion.div>
            )}
            <div ref={messagesEndRef} />
          </div>
        )}
      </div>

      {/* Input Area */}
      <div className="border-t border-slate-200 p-6 bg-slate-50">
        <div className="flex gap-3">
          <input
            type="text"
            value={input}
            onChange={(e) => setInput(e.target.value)}
            onKeyPress={(e) => e.key === 'Enter' && handleSendMessage()}
            placeholder="Ask a question about your documents..."
            disabled={isLoading || documents.length === 0}
            className="flex-1 px-4 py-3 border border-slate-200 rounded-lg focus:border-blue-500 focus:outline-none focus:ring-2 focus:ring-blue-500/20 disabled:bg-slate-100"
          />
          <motion.button
            whileHover={{ scale: 1.05 }}
            whileTap={{ scale: 0.95 }}
            onClick={handleSendMessage}
            disabled={isLoading || !input.trim() || documents.length === 0}
            className="px-4 py-3 bg-blue-600 text-white rounded-lg hover:bg-blue-700 disabled:bg-slate-400 transition-colors"
          >
            <Send size={20} />
          </motion.button>
        </div>

        {documents.length === 0 && (
          <p className="text-xs text-slate-500 mt-3">📤 Upload documents to start asking questions</p>
        )}
      </div>
    </div>
  );
};

export default ChatPanel;
'''

print(chat_panel_code)

## 9. React Router Configuration

Navigation setup with routes for dashboard and group workspaces.

In [ ]:
# File: src/main.tsx - Updated entry point with Router
router_config = '''
import React from 'react'
import ReactDOM from 'react-dom/client'
import { BrowserRouter as Router, Routes, Route, Navigate } from 'react-router-dom'
import App from './App'
import { GroupProvider } from './context/GroupContext'
import { GroupDashboard } from './components/GroupDashboard'
import { GroupPage } from './components/GroupPage'
import './index.css'

ReactDOM.createRoot(document.getElementById('root')!).render(
  <React.StrictMode>
    <GroupProvider>
      <Router>
        <Routes>
          {/* Dashboard - List all groups */}
          <Route path="/" element={<GroupDashboard />} />

          {/* Group Workspace - Single group with RAG chat */}
          <Route path="/group/:groupId" element={<GroupPage />} />

          {/* Fallback */}
          <Route path="*" element={<Navigate to="/" replace />} />
        </Routes>
      </Router>
    </GroupProvider>
  </React.StrictMode>,
)
'''

print(router_config)

# Also show main app structure update
app_structure = '''
# File: src/App.tsx - Can now remain as a legacy component OR be replaced with shell
# Option A: Keep as legacy - only shown when no groups exist
# Option B: Simplify - remove from active routes, archive for reference

# Updated App.tsx role (Option A):
# - Only mounted if GroupContext has no groups (backwards compatibility)
# - Users can upload single file, converts to group automatically
# - Smooth migration path from old UI to new workspace system

# Key changes for App.tsx if keeping:
1. Wrap in conditional: only show if state.groups.length === 0
2. Add action: auto-create group from first file upload
3. Redirect to /group/:groupId after upload
4. Optional: Show "Switch to workspace view" button
'''

print(app_structure)

## 10. Implementation Roadmap & Summary

Step-by-step guide for implementing the NotebookLM-style workspace redesign.

In [ ]:
# Implementation Guide
implementation_guide = '''
╔════════════════════════════════════════════════════════════════════════╗
║           WORKSPACE REDESIGN - IMPLEMENTATION ROADMAP                  ║
╚════════════════════════════════════════════════════════════════════════╝

## PHASE 1: Infrastructure (2-3 hours)
───────────────────────────────────────────────────────────────────────

1️⃣ Create GroupContext & Provider
   ✓ File: src/context/GroupContext.tsx
   ✓ Features: Reducer pattern, localStorage sync, auto-load last group
   ✓ Exports: GroupContext, GroupProvider, useGroupContext hook
   ✓ Test: Can create, update, delete groups; persists to localStorage

2️⃣ Install required dependencies (if needed)
   npm install react-router-dom date-fns
   
3️⃣ Update main.tsx
   ✓ Wrap app with GroupProvider
   ✓ Add Router with 2 routes: / and /group/:groupId
   ✓ Remove old App.tsx from main routes (keep as backup)


## PHASE 2: Dashboard Components (4-5 hours)
───────────────────────────────────────────────────────────────────────

4️⃣ Create GroupDashboard Component
   ✓ File: src/components/GroupDashboard.tsx
   ✓ Features: Group listing, search, delete, create modal
   ✓ State: searchQuery, showCreateModal
   ✓ Styling: Grid layout, card animations
   ✓ Test: Can search groups, delete group, open create modal

5️⃣ Create GroupCard Component
   ✓ File: src/components/GroupCard.tsx
   ✓ Features: Display group info, doc count, creation date
   ✓ Animations: Hover effects, click feedback
   ✓ Exports: GroupCard as default export
   ✓ Test: Shows correct doc count, formatted dates, delete button

6️⃣ Create CreateGroupModal Component
   ✓ File: src/components/CreateGroupModal.tsx
   ✓ Features: Form validation, error messages, loading state
   ✓ Actions: Dispatch CREATE_GROUP to context
   ✓ Test: Validates name (min 3 chars), creates with description


## PHASE 3: Workspace Components (5-6 hours)
───────────────────────────────────────────────────────────────────────

7️⃣ Create DocumentSidebar Component
   ✓ File: src/components/DocumentSidebar.tsx
   ✓ Features: Dropzone upload, doc list, processing status
   ✓ Props: documents[], onFileUpload(), isProcessing
   ✓ Test: Can drag-drop files, shows doc list, displays processing


8️⃣ Create ChatPanel Component
   ✓ File: src/components/ChatPanel.tsx
   ✓ Features: Message display, input area, action buttons
   ✓ Methods: Q&A, Summarize, Key Points, Compare, Explain
   ✓ Reuses: ChatEngine methods from existing code
   ✓ Test: Can send messages, trigger actions, show responses

   📝 Helper: Create src/components/MessageItem.tsx
   ✓ Displays message with role-based styling
   ✓ Props: Message interface, isLast boolean
   ✓ Features: Timestamp, optional metadata badge


9️⃣ Create GroupPage Component
   ✓ File: src/components/GroupPage.tsx
   ✓ Features: Two-panel layout (sidebar + chat)
   ✓ State: sidebarOpen, isProcessing, currentGroup
   ✓ Methods: Handle file upload, dispatch ADD_DOCUMENT
   ✓ Mobile: Toggle sidebar, responsive layout
   ✓ Test: Can upload files, displays in sidebar, send chat messages


## PHASE 4: Integration & Polish (2-3 hours)
───────────────────────────────────────────────────────────────────────

🔟 Integrate with existing services
   ✓ ChatEngine: Already supports multi-file analysis ✓
   ✓ DocumentProcessor: Already handles extraction ✓
   ✓ VectorStore: Already supports project isolation ✓
   
   📋 Create helper utilities:
   ✓ File: src/utils/format.ts
   ✓ Functions: formatBytes(), formatDistanceToNow()
   ✓ Reuse: date-fns library for date formatting

1️⃣1️⃣ Add utility helpers
   ✓ File: src/utils/file.ts
   ✓ Functions: hashFile(), getFileIcon(), validateFile()
   ✓ Features: File validation, type detection, hash generation


1️⃣2️⃣ Update package.json
   ✓ Install: react-router-dom@latest
   ✓ Install: date-fns for date formatting
   ✓ Update: vite.config.ts if needed


## PHASE 5: Testing & Deployment (2-3 hours)
───────────────────────────────────────────────────────────────────────

1️⃣3️⃣ Functional tests
   ✓ Create group → navigate to workspace
   ✓ Upload document → appears in sidebar
   ✓ Send Q&A → gets response from ChatEngine
   ✓ Click Summarize → shows summary
   ✓ Delete group → removes from list, redirect to /
   ✓ Refresh page → auto-loads last opened group
   ✓ Search groups → filters correctly


1️⃣4️⃣ UI/UX polish
   ✓ Loading states: Add skeleton loader for groups
   ✓ Error handling: Toast notifications for failures
   ✓ Accessibility: Tab navigation, ARIA labels
   ✓ Mobile: Test responsive layout on <768px


1️⃣5️⃣ Build & deploy
   npm run build
   npm run preview


═════════════════════════════════════════════════════════════════════════

## Estimated Timeline

Phase 1 (Infrastructure):      2-3 hours
Phase 2 (Dashboard):           4-5 hours
Phase 3 (Workspace):           5-6 hours
Phase 4 (Integration):         2-3 hours
Phase 5 (Testing/Deploy):      2-3 hours
                              ──────────
TOTAL:                        15-20 hours

## Key Design Principles Used

1. ✅ **NotebookLM Pattern**: Workspace-first, groups as containers
2. ✅ **State Management**: React Context API with reducer pattern
3. ✅ **Persistence**: Automatic localStorage sync for offline access
4. ✅ **Reusability**: Leverage existing ChatEngine, VectorStore, services
5. ✅ **Performance**: Two-panel layout with lazy loading
6. ✅ **Mobile First**: Responsive design with toggleable sidebar
7. ✅ **User Experience**: Auto-load last group, search, quick actions

## Migration Strategy

For existing users:
1. Keep old App.tsx as fallback for backward compatibility
2. Detect if groups exist in localStorage
3. If no groups: show legacy UI, auto-create group on first upload
4. If groups exist: show new workspace dashboard
5. Optional: Add "Migrate to workspace" button for power users

This ensures zero disruption while guiding users to new system.
'''

print(implementation_guide)